In [55]:
!pip install -q pyspark
!pip install -U -q PyDrive

In [56]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]


In [57]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg
import time

In [58]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StudentDepressionAnalysis") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("✅ Spark Session Started")
print("Spark Version:", spark.version)


✅ Spark Session Started
Spark Version: 3.5.1


In [ ]:
from google.colab import files
print("Upload  Student Depression Dataset CSV")
uploaded = files.upload()

fname = list(uploaded.keys())[0]
df = spark.read.csv(fname, header=True, inferSchema=True)
print("✅ Data loaded successfully!")
df.printSchema()
df.show(5, truncate=False)

Upload  Student Depression Dataset CSV


In [ ]:
df.count
print("Total records:", df.count())


print("Total columns:", len(df.columns))




In [ ]:
df = df.na.drop()

In [ ]:
for old_col in df.columns:
    new_col = old_col.strip() \
                    .replace(" ", "_") \
                    .replace("/", "_") \
                    .replace("-", "_") \
                    .replace("?", "")
    df = df.withColumnRenamed(old_col, new_col)

df.createOrReplaceTempView("students")

In [ ]:
print(df.columns)

In [ ]:
df_filtered = df.filter(
    "Age > 18 AND City IS NOT NULL AND Dietary_Habits IS NOT NULL"
)


df_filtered.select("City", "Dietary_Habits").show(20, False)

In [ ]:
spark.sql("""
SELECT
  COUNT(*) AS total_students,
  AVG(Sleep_Duration) AS avg_sleep,
  AVG(CGPA) AS avg_cgpa,
  AVG(Work_Study_Hours) AS avg_study_hours
FROM students
""").show()


In [ ]:
spark.sql("""
SELECT Depression AS Depression_Status,
       COUNT(*) AS count,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM students), 2) AS percentage
FROM students
GROUP BY Depression
""").show()


In [ ]:
numeric_cols = [f.name for f in df.schema.fields if str(f.dataType) in ('IntegerType','DoubleType')]
print("Numeric Columns:", numeric_cols)

for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        c1, c2 = numeric_cols[i], numeric_cols[j]
        corr_value = df.stat.corr(c1, c2)
        print(f"Correlation between {c1} and {c2}: {corr_value:.2f}")


In [ ]:
recommendations = spark.sql("""
SELECT
  id AS ID,
  Gender,
  Age,
  Sleep_Duration AS Sleep_Duration,
  Work_Pressure AS Work_Pressure,
  Work_Study_Hours AS Study_Hours,
  Academic_Pressure,
  CASE
    WHEN Sleep_Duration < 5 THEN 'Recommend Sleep Hygiene Program'
    WHEN Work_Pressure > 3 THEN 'Recommend Stress Counseling'
    WHEN Academic_Pressure > 3 THEN 'Academic Support and Mentorship'
    ELSE 'Regular Wellness Check'
  END AS Recommended_Program
FROM students
""")

recommendations.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "Sleep_Duration_Numeric",
    when(col("Sleep_Duration") == "Less than 5 hours", 4.0)
    .when(col("Sleep_Duration") == "5-6 hours", 5.5)
    .when(col("Sleep_Duration") == "7-8 hours", 7.5)
    .when(col("Sleep_Duration") == "More than 8 hours", 9.0)
    .otherwise(None)
)

df = df.withColumn(
    "RiskScore",
    col("Sleep_Duration_Numeric") * -1 +
    col("Work_Pressure") * 2 +
    col("Academic_Pressure") * 2 +
    col("Work_Study_Hours") * 0.5
)

df.createOrReplaceTempView("students_risk")

risk_recommendations = spark.sql("""
SELECT *,
  CASE
    WHEN RiskScore >= 10 THEN 'High Risk: Intensive Counseling + Mentorship'
    WHEN RiskScore >= 5 THEN 'Moderate Risk: Stress & Academic Support'
    ELSE 'Low Risk: Regular Wellness Check'
  END AS Recommended_Program
FROM students_risk
""")

risk_recommendations.show(10, truncate=False)

Sometimes an error occurs when adding the lib, so I had to add the lib separately.

In [ ]:
rdd = risk_recommendations.rdd

rdd_mapped = rdd.map(lambda row: (
    row.id,
    row.RiskScore,
    "High Risk" if row.RiskScore is not None and row.RiskScore >= 10 else "Moderate Risk" if row.RiskScore is not None and row.RiskScore >= 5 else "Low Risk"
))


print("RDD Mapping")
for item in rdd_mapped.take(10):
    print(item)

In [ ]:
spark.sql("""
SELECT
    id,
    Gender,
    Work_Study_Hours,
    Sleep_Duration,
    CASE
        WHEN Work_Study_Hours > 8 AND Sleep_Duration < 6 THEN 'Reduce Study Load + Sleep Hygiene'
        WHEN Work_Study_Hours > 8 THEN 'Reduce Study Load'
        WHEN Sleep_Duration < 6 THEN 'Improve Sleep Schedule'
        ELSE 'Keep Current Routine'
    END AS Recommended_Program
FROM students
""").show()


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import percent_rank

In [ ]:
windowSpec = Window.orderBy(col("Work_Pressure").desc())


In [ ]:
df = df.withColumn("WorkPressureRank", percent_rank().over(windowSpec))

In [ ]:
recommendations_percentile = df.select(
    "ID",
    "Gender",
    "Age",
    "Work_Pressure",
    "Sleep_Duration",
    "Academic_Pressure",
    expr("""
        CASE
            WHEN WorkPressureRank > 0.75 THEN 'High Stress Counseling'
            WHEN Sleep_Duration < 5 THEN 'Sleep Hygiene Program'
            ELSE 'Regular Wellness Check'
        END AS Recommended_Program
    """)
)

recommendations_percentile.show(10, truncate=False)

In [ ]:
from sklearn.metrics import precision_score
import pandas as pd

In [ ]:
recommendations_pd = pd.DataFrame({
    "ID": [1, 2, 3, 4],
    "Recommended_Program": ["High Stress Counseling", "Regular Wellness Check", "Sleep Hygiene Program", "Regular Wellness Check"]
})


recommendations_pd['Actual_Program'] = ["High Stress Counseling", "Sleep Hygiene Program", "Sleep Hygiene Program", "Regular Wellness Check"]

print(recommendations_pd)

In [ ]:


actual = recommendations_pd['Actual_Program']
predicted = recommendations_pd['Recommended_Program']

precision = precision_score(actual, predicted, average='macro')
print(f"Precision: {precision:.4f}")


In [ ]:
student_recs = {
    1: ['Regular Wellness Check','Sleep Hygiene Program','High Stress Counseling'],
    2: ['High Stress Counseling','Regular Wellness Check'],
    3: ['Sleep Hygiene Program','High Stress Counseling']
}

student_actual = {
    1: 'Sleep Hygiene Program',
    2: 'High Stress Counseling',
    3: 'Sleep Hygiene Program'
}

mrr = 0
for student, recs in student_recs.items():
    actual_item = student_actual[student]
    for rank, item in enumerate(recs, start=1):
        if item == actual_item:
            mrr += 1/rank
            break

mrr /= len(student_actual)
print(f"Mean Reciprocal Rank (MRR): {mrr:.4f}")

In [ ]:
coverage = recommendations_pd['Recommended_Program'].value_counts(normalize=True) * 100
print("Recommendation Coverage (%):\n", coverage)


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(recommendations_pd['Actual_Program'],
                            recommendations_pd['Recommended_Program']))


In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql.functions import when, col

In [ ]:
spark = SparkSession.builder.appName("StudentRecommendationAnalysis").getOrCreate()

In [ ]:
df = spark.read.option("header", True).csv("Student Depression Dataset.csv")

In [ ]:
print("Columns in dataset:", df.columns)

In [ ]:
df = df.withColumn(
    "Recommended_Program",
    when(col("Depression") == 1, "Counseling") \
    .when(col("Depression") == 0, "No Action Needed") \
    .otherwise("Wellness Program")
)

In [ ]:
program_counts = df.groupBy("Recommended_Program").count().orderBy(col("count").desc())
program_counts.show()

In [ ]:
program_counts_pd = program_counts.toPandas()

In [ ]:
plt.figure(figsize=(10,6))
plt.bar(program_counts_pd['Recommended_Program'], program_counts_pd['count'], color='skyblue')
plt.xlabel("Recommended Program")
plt.ylabel("Number of Students")
plt.title("Student Distribution by Recommended Program")
plt.xticks(rotation=30)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
plt.pie(
    program_counts_pd['count'],
    labels=program_counts_pd['Recommended_Program'],
    autopct='%1.1f%%',
    startangle=140,
    colors=['lightcoral', 'lightskyblue', 'lightgreen']
)

In [ ]:
gender_depression = df.groupBy("Gender", "Depression").count().toPandas()

import seaborn as sns

plt.figure(figsize=(8,5))
sns.barplot(data=gender_depression, x="Gender", y="count", hue="Depression")
plt.title("Depression Distribution by Gender")
plt.show()


In [ ]:
from pyspark.sql.functions import when


df_numeric_sleep = df.withColumn(
    "Sleep_Duration_Numeric",
    when(col("Sleep Duration") == "Less than 5 hours", 4.0)
    .when(col("Sleep Duration") == "5-6 hours", 5.5)
    .when(col("Sleep Duration") == "7-8 hours", 7.5)
    .when(col("Sleep Duration") == "More than 8 hours", 9.0)
    .otherwise(None)
)

sleep_depression = df_numeric_sleep.groupBy("Depression").avg("Sleep_Duration_Numeric").toPandas()

plt.figure(figsize=(6,5))
plt.bar(sleep_depression['Depression'], sleep_depression['avg(Sleep_Duration_Numeric)'], color='green')
plt.xlabel("Depression Level")
plt.ylabel("Average Sleep Duration (hours)")
plt.title("Average Sleep Duration by Depression Level")
plt.xticks([0, 1], ['No Depression', 'Depression'])
plt.show()